# 02 — Build features

Sweeps the raw `.mat` parts and produces:
- `features.parquet` — engineered features + labels (consumed by RF and SVM).
- `signals.h5` — raw `(N, 2, 1000)` PPG+ECG segments for the CNN. **Off by default** — flip `WRITE_SIGNAL_CACHE = True` if/when you want to train the CNN.

Parallelized via `joblib` over records — uses every CPU core on the box.

## Sampling strategy

A single record can contain 5–10 minutes of recording, which yields 35–70 nearly-identical 8-second segments (same subject, same physiological state). Treating those as independent training examples inflates effective sample size and biases the model toward subjects with longer recordings.

To address this we cap **segments per record** (default `MAX_SEGMENTS_PER_RECORD = 15`, evenly spaced across the recording). This:
- Decorrelates the training set temporally
- Preserves all 942 unique subjects
- Cuts the dataset to ~150k segments (vs ~600k naive) — RF still saturates well above this
- Makes SVM training tractable (its O(n²–n³) cost is the bottleneck for the partner's model)

Segments still pass `is_valid_segment` QC and `label_segment` peak-detection checks before being written, so unhealthy windows are dropped on top of the cap.

In [ ]:
PARTS = [1, 2, 3, 4]
MAX_RECORDS = None              # None = all records (~12k); set to e.g. 50 for a smoke test
MAX_SEGMENTS_PER_RECORD = 15    # evenly-spaced cap per record (see markdown above)
WRITE_SIGNAL_CACHE = True       # Also writes signals.h5 — required by the CNN in nb 05
BATCH_SIZE = 32                 # records dispatched to joblib per chunk
N_JOBS = -1                     # -1 = all cores

import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from joblib import Parallel, delayed
from tqdm.auto import tqdm

from bme_ml import WINDOW_SAMPLES
from bme_ml.paths import setup_paths
from bme_ml.data_loader import iter_records
from bme_ml.pipeline import process_record

if WRITE_SIGNAL_CACHE:
    import h5py

paths = setup_paths()
print('parquet ->', paths.features_parquet)
if WRITE_SIGNAL_CACHE:
    print('signals ->', paths.signals_h5)
else:
    print('(signals.h5 disabled — flip WRITE_SIGNAL_CACHE=True to enable for CNN)')

In [ ]:
def gen_records():
    n = 0
    for part in PARTS:
        for rec_idx, rec in enumerate(iter_records(paths.raw / f'Part_{part}.mat')):
            yield part, rec_idx, rec
            n += 1
            if MAX_RECORDS is not None and n >= MAX_RECORDS:
                return

def chunked(it, n):
    batch = []
    for item in it:
        batch.append(item)
        if len(batch) == n:
            yield batch
            batch = []
    if batch:
        yield batch

In [ ]:
parquet_writer = None

if WRITE_SIGNAL_CACHE:
    h5 = h5py.File(paths.signals_h5, 'w')
    sig_dset = h5.create_dataset(
        'signals', shape=(0, 2, WINDOW_SAMPLES), maxshape=(None, 2, WINDOW_SAMPLES),
        dtype='float32', chunks=(256, 2, WINDOW_SAMPLES), compression='gzip', compression_opts=4,
    )
    label_dset = h5.create_dataset(
        'label_binary', shape=(0,), maxshape=(None,), dtype='int8', chunks=(4096,),
    )
    subj_dset = h5.create_dataset(
        'subject_id', shape=(0,), maxshape=(None,),
        dtype=h5py.string_dtype(encoding='utf-8'), chunks=(4096,),
    )

n_total = 0
try:
    for batch in tqdm(chunked(gen_records(), BATCH_SIZE), desc='batches'):
        results = Parallel(n_jobs=N_JOBS, batch_size=1)(
            delayed(process_record)(part, rec_idx, rec, MAX_SEGMENTS_PER_RECORD)
            for part, rec_idx, rec in batch
        )
        rows = [r for rows, _ in results for r in rows]
        sigs = [s for _, segs in results for s in segs]
        if not rows:
            continue
        # Parquet write.
        df = pd.DataFrame(rows)
        table = pa.Table.from_pandas(df, preserve_index=False)
        if parquet_writer is None:
            parquet_writer = pq.ParquetWriter(paths.features_parquet, table.schema)
        parquet_writer.write_table(table)
        # H5 append (signals + labels + subject_id, all in lockstep with parquet rows).
        if WRITE_SIGNAL_CACHE:
            sigs_arr = np.stack(sigs)
            labels_arr = np.array([r['label_binary'] for r in rows], dtype=np.int8)
            subj_arr = np.array([f"p{r['part']}_r{r['record']}" for r in rows], dtype=object)
            old = sig_dset.shape[0]
            new = old + len(sigs_arr)
            sig_dset.resize(new, axis=0); sig_dset[old:new] = sigs_arr
            label_dset.resize(new, axis=0); label_dset[old:new] = labels_arr
            subj_dset.resize(new, axis=0); subj_dset[old:new] = subj_arr
        n_total += len(rows)
finally:
    if parquet_writer is not None:
        parquet_writer.close()
    if WRITE_SIGNAL_CACHE:
        h5.close()

print(f'wrote {n_total} segments')

In [ ]:
# Sanity check.
df = pd.read_parquet(paths.features_parquet)
print('rows :', len(df))
print('label balance:')
print(df['label_binary'].value_counts(normalize=True))
if WRITE_SIGNAL_CACHE:
    import h5py
    with h5py.File(paths.signals_h5, 'r') as f:
        print('signals h5 :', f['signals'].shape, f['signals'].dtype)
        print('labels h5  :', f['label_binary'].shape)
        print('subjects h5:', f['subject_id'].shape)
        assert f['signals'].shape[0] == len(df), 'parquet/h5 row count mismatch'
df.head()